# NewsQA RAG - Phase 1 Retrieval Tournament (Kaggle T4 x2)
End-to-end resumable benchmark. Original questions select the winner; resolved questions are paired supplementary analysis.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time
REPO_URL = 'https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT = 'SET_TO_COMMIT_CONTAINING_KAGGLE_RUNNER'  # required immutable SHA
HF_REPO_ID = 'ThomasAnderson2009/newsqa-rag-evaluation'
HF_REVISION = 'v1.0.0'
FAST_MODE = False
RUN_LATENCY_CALIBRATION = True
LATENCY_REPEATS, LATENCY_QUESTIONS = 3, 100
RESTORE_CHECKPOINT = ''
KAGGLE_WORKING = Path('/kaggle/working')
PROJECT_ROOT = KAGGLE_WORKING / 'Text-Mining---NewsQA-RAG'
WORK_ROOT = KAGGLE_WORKING / 'newsqa_phase1'
RESULTS = WORK_ROOT / 'results'
assert REPO_COMMIT != 'SET_TO_COMMIT_CONTAINING_KAGGLE_RUNNER', 'Set REPO_COMMIT before Save & Run All'


## 1. Secrets, GPUs, repository and dependencies

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_HOME'] = str(KAGGLE_WORKING / 'hf_cache')
os.environ.update({'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1'})
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--all','--tags'],cwd=PROJECT_ROOT,check=True)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, 'Select Kaggle GPU T4 x2'
for i in range(2): print(i,torch.cuda.get_device_name(i),round(torch.cuda.get_device_properties(i).total_memory/2**30,1),'GiB')
print('Pinned commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_ROOT,text=True).strip())


In [ ]:
import pandas as pd
from IPython.display import display, Image
def run_driver(stage):
    cmd=[sys.executable,'scripts/run_phase1_kaggle.py','--repo-id',HF_REPO_ID,'--revision',HF_REVISION,'--work-root',str(WORK_ROOT),'--stop-after',stage]
    if FAST_MODE: cmd.append('--fast')
    if RESTORE_CHECKPOINT and stage=='round1' and not WORK_ROOT.exists(): cmd += ['--restore-checkpoint',RESTORE_CHECKPOINT]
    print('$',' '.join(cmd)); subprocess.run(cmd,cwd=PROJECT_ROOT,check=True)
def show_csv(name,sort='retrieval.mrr@5.mean'):
    frame=pd.read_csv(RESULTS/name)
    if sort in frame: frame=frame.sort_values(sort,ascending=False)
    display(frame); return frame
def paired(frame):
    keys=[k for k in ['index','retriever','reranker','reranker_model','partition'] if k in frame]
    values=[k for k in ['retrieval.mrr@5.mean','retrieval.hit_rate@5.mean','retrieval.ndcg@5.mean'] if k in frame]
    return frame.pivot_table(index=keys,columns='variant',values=values).reset_index()


## 2. Dataset preparation and Round 1: four dense + four sparse profiles

In [ ]:
started=time.time(); run_driver('round1')
round1=show_csv('round1.csv'); display(paired(round1))
print('Minutes:',round((time.time()-started)/60,1)); display(json.loads((RESULTS/'round1_winners.json').read_text()))


## 3. Round 2: Best Dense / Best Sparse / Hybrid x three rerankers

In [ ]:
started=time.time(); run_driver('round2')
round2=show_csv('round2.csv'); display(paired(round2))
print('Minutes:',round((time.time()-started)/60,1)); display(json.loads((RESULTS/'winner_lock.json').read_text()))


## 4. Round 3: chunk 256/32, 512/64 and 1024/128

In [ ]:
started=time.time(); run_driver('round3')
round3=show_csv('round3.csv'); display(paired(round3))
print('Minutes:',round((time.time()-started)/60,1)); display(json.loads((RESULTS/'winner_lock.json').read_text()))


## 5. Locked final-test: original and resolved

In [ ]:
started=time.time(); run_driver('final')
final_test=show_csv('final_test.csv'); display(paired(final_test))
print('Minutes:',round((time.time()-started)/60,1))


## 6. Serial cache-free latency calibration

In [ ]:
if RUN_LATENCY_CALIBRATION and not FAST_MODE:
    specs=sorted((WORK_ROOT/'specs').glob('round*.yaml'))
    cmd=[sys.executable,'scripts/calibrate_phase1_latency.py',*map(str,specs),'--output',str(RESULTS/'latency_calibration.csv'),'--repeats',str(LATENCY_REPEATS),'--n-eval',str(LATENCY_QUESTIONS)]
    subprocess.run(cmd,cwd=PROJECT_ROOT,check=True,env={**os.environ,'CUDA_VISIBLE_DEVICES':'0'})
    show_csv('latency_calibration.csv',sort='latency.total.p50_ms')
else: print('Latency calibration skipped')


## 7. Figures and downloadable Kaggle outputs

In [ ]:
subprocess.run([sys.executable,'scripts/export_phase1_results.py','--experiments-root',str(WORK_ROOT/'experiments'),'--output-dir',str(RESULTS)],cwd=PROJECT_ROOT,check=True)
for figure in sorted((RESULTS/'figures').glob('*.png')):
    print(figure.name); display(Image(filename=str(figure)))
for path in sorted(RESULTS.rglob('*')):
    if path.is_file(): print(path.relative_to(RESULTS),round(path.stat().st_size/2**20,2),'MiB')
print('Results:',KAGGLE_WORKING/'phase1_results_bundle.zip')
print('Checkpoint:',KAGGLE_WORKING/'phase1_checkpoint.tar')
